# Neural Networks 2

Today we'll talk about backpropagation and more complex neural networks.



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)


In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/intro-stat-learning/ISLP/main/ISLP/data/Credit.csv")

In [3]:
df = df[["Income", "Limit", "Balance"]]
df = df.rename(columns=str.lower)
df = (df - df.mean()) / df.std()
df.shape

(400, 3)

## Where we ended last time

We finished our last lecture with this neural network:

<img src="one_feature_one_neuron.png" width="1000">

Pay attention to the notation - the superscript in parentheses `(L)` indicates the layer to which the parameter or variable belongs.


The loss of this network is:

$$
\mathcal{L}\bigl(w^{(1)},b^{(1)},w^{(2)},b^{(2)}\bigr)
  = \sum_{i=1}^{m} \bigl(z^{(2)}_i - y_i\bigr)^2
  = \sum_{i=1}^{m}
    \left(
      w^{(2)}\,\sigma\!\bigl(w^{(1)}x_i + b^{(1)}\bigr)
      + b^{(2)}
      - y_i
    \right)^{2}
$$

W and b in both layers are scalars. The index in the brackets indicates which layer the variable belongs to.


In [23]:
X = df["limit"].values
y = df["balance"].values
X.shape

(400,)

In [24]:
# activation
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# loss
def sum_of_squares(a, y):
    return np.sum((a - y) ** 2)


In [25]:
w1 = np.float32(1)
b1 = np.float32(1)
w2 = np.float32(1)
b2 = np.float32(1)


In [26]:
z1 = X * w1 + b1
a1 = sigmoid(z1)
z2 = a1 * w2 + b2
L = sum_of_squares(z2, y)
L

np.float64(1444.3526409019069)

What we have done with the code above is known as the *forward pass* in the neural network - we've passed the data through the network once and calculated the loss.


Now we want to perform gradient descent, for which we have to find these four gradients:

$$
\frac{\partial\mathcal{L}}{\partial w^{(2)}};
\frac{\partial\mathcal{L}}{\partial b^{(2)}};
\frac{\partial\mathcal{L}}{\partial w^{(1)}};
\frac{\partial\mathcal{L}}{\partial b^{(1)}}
$$


The problem is that network now contains functions of functions - $z^{(2)}$ is a function of $a^{(1)}$, $w^{(2)}$ and $b^{(2)}$, and $a^{(1)}$ is itself a function of $w^{(1)}$ and $b^{(1)}$. When we have to find partial derivatives of a composite function $f(g(x))$, we turn to the chain rule.

The sigmoid activation function has a convenient derivative:

$$
\sigma'(z) = \sigma(z)\bigl(1 - \sigma(z)\bigr)
$$

Since $a^{(1)}_i = \sigma(z^{(1)}_i)$, we can compute $\sigma'(z^{(1)}_i)$ by reusing the activation we already calculated in the forward pass:

$$
\sigma'(z^{(1)}_i) = a^{(1)}_i \bigl(1 - a^{(1)}_i\bigr)
$$

The formulas for the gradients are:

$$

\frac{\partial\mathcal{L}}{\partial w^{(2)}} = 2\sum_{i=1}^{m}\!\Bigl( w^{(2)}a^{(1)}_i  + b^{(2)} - y_i \Bigr)\,a^{(1)}_i
\\
\frac{\partial\mathcal{L}}{\partial b^{(2)}} = 2\sum_{i=1}^{m}\!\Bigl( w^{(2)}a^{(1)}_i  + b^{(2)} - y_i \Bigr)
\\
\frac{\partial\mathcal{L}}{\partial w^{(1)}} = 2\sum_{i=1}^{m}\!\Bigl( w^{(2)}a^{(1)}_i  + b^{(2)} - y_i \Bigr)\,w^{(2)}\,\sigma'(z^{(1)}_i)\,x_i
\\
\frac{\partial\mathcal{L}}{\partial b^{(1)}} = 2\sum_{i=1}^{m}\!\Bigl( w^{(2)}a^{(1)}_i  + b^{(2)} - y_i \Bigr)\,w^{(2)}\,\sigma'(z^{(1)}_i)

$$

Computing these gradients is what we call the backward pass, or backpropagation.

We can actually ignore the 2 in the front in the formulas above, since it just scales the gradients, and we will scale them anyways using the learning rate.



In [27]:
def d_sigmoid(z):
    a = sigmoid(z)
    return a * (1 - a)


### Task 1

Finish the code for the backward pass below.


In [28]:
# Initialize weights
w1 = np.float32(2)
b1 = np.float32(2)
w2 = np.float32(2)
b2 = np.float32(2)

lr = 0.001

# forward pass
z1 = X * w1 + b1
a1 = sigmoid(z1)
z2 = a1 * w2 + b2

# loss
L = sum_of_squares(z2, y)

# backward pass (backpropagation)
err = z2 - y
grad_w2 = ...
grad_b2 = ...
grad_w1 = ...
grad_b1 = ...


### Task 2

Complete the code below using the code you wrote for the backward pass + implementing the gradient descent.

Print out the weights and visualize the function that your model is fitting to the data.

In [31]:
# Initialize weights
w1 = np.float32(2)
b1 = np.float32(2)
w2 = np.float32(2)
b2 = np.float32(2)

lr = 0.001

for i in range(100000):
    # forward pass
    z1 = X * w1 + b1
    a1 = sigmoid(z1)
    z2 = a1 * w2 + b2

    # loss
    L = sum_of_squares(z2, y)

    # backprop

    # gradient descent
    


## More neurons

Let's stick with one feature, but add more neurons to that hidden layer. The complication now is that we have more weights, and even though an individual weight is still scalar, writing backprop for every individual weight will quickly become unmanageable and inefficient. Therefore, from now on we will perform matrix operations.

<img src="one_feature_two_neurons.png" width="1000">



In [35]:
X = df[["limit"]].values
y = df[["balance"]].values

### Task 3

Implement the forward pass of the illustrated network.

1. Initialize the weights in the right shape
2. Implement the forward pass

The cell below illustrates the relevant numpy operation you will need — matrix multiplication via the `@` operator. Note how the shapes of the two matrices have to align for the multiplication to work.

Tip: after performing an operation, print out the shape of the resulting matrix using, e.g. `z1.shape`

In [36]:
# Relevant numpy operations
X1 = np.ones((3, 3))
X2 = np.ones((3, 1))
X1 @ X2


array([[3.],
       [3.],
       [3.]])

(400, 1)
(400, 2)
(400, 2)
(400, 1)


### Task 4

Implement a single step of gradient descent for the neural network. Since we are no longer working with scalars, our gradient calculations became a little more complex as well. The exact formulas are given below.

Run the cell that performs gradient descent several times, while printing out the weights. What do you notice?

In [39]:
# initialize weights
W1 = np.ones((1, 2))
b1 = np.ones((2))
W2 = np.ones((2, 1))
b2 = np.ones((1))


In [ ]:
# forward pass
z1 = X @ W1 + b1
a1 = sigmoid(z1)
z2 = a1 @ W2 + b2

lr = 0.0001

# Calculate gradients
err = z2 - y
grad_W2 = a1.T @ err
grad_b2 = np.sum(err, axis=0)
grad_W1 = X.T @ ((err @ W2.T) * d_sigmoid(z1))
grad_b1 = np.sum((err @ W2.T) * d_sigmoid(z1), axis=0)

# Update weights

# Print weights


346.1572546630355


(array([[0.35129489],
        [0.35129489]]),
 array([-0.19248814]),
 array([[1.1056548, 1.1056548]]),
 array([0.82613264, 0.82613264]))

### Task 5

Implement a function `train_network` that performs gradient descent of the neural network. Use it to perform gradient descent, then print out the weights. What do you notice?


In [ ]:
def train_network(X, y, W1, b1, W2, b2, lr=0.0001, n_iter=10000):
    for _ in range(n_iter):
        # forward pass

        # backpropagation

        # gradient descent

    return W1, b1, W2, b2

W1, b1, W2, b2 = train_network(X, y, W1, b1, W2, b2)

W1, b1, W2, b2


(array([[1.33850709, 1.33850709]]),
 array([-0.10240531, -0.10240531]),
 array([[1.74728421],
        [1.74728421]]),
 array([-1.63150005]))

### Task 6

Fix the problem above by choosing the weights randomly.

Then, find the optimal weights using the `train_network` function. Lastly, make predictions using the new weights and plot the fit.

In [72]:
# hint
np.random.randn(3, 1)
np.random.randn(1)

array([0.76743473])

### Task 7

Implement a network with 5 neurons in the hidden layer, and run gradient descent for 20k iterations. Plot the resulting fit.


With 5 hidden neurons the network now combines 5 sigmoids, and the fitted curve bends to follow the data much more closely than the single-neuron version could. This is the universal-approximation idea in practice — more hidden units mean more flexible fits.

## Two input features, single neuron


<img src="two_features_one_neuron.png" width="1000">

Up to now we've kept a single input feature and varied the number of hidden neurons. Now we vary the other axis — two input features, back to just one hidden neuron. This is where the matrix-form code pays off: train_network doesn't need to change, only the shape of W1 does.




### Task 8

Train the neural network shown above:
1. select the features
2. initialize the weights
3. implement the forward pass
4. train the network using the function you implemented

Use random initialization for the weights. 


Now let's look at what kind of function we're fitting. We do not need to make any changes to backprop.

In [28]:
import plotly.graph_objects as go

x_range = np.linspace(X[:, 0].min(), X[:, 0].max(), 100)
y_range = np.linspace(X[:, 1].min(), X[:, 1].max(), 100)
x_mesh, y_mesh = np.meshgrid(x_range, y_range)

X_mesh = np.column_stack((x_mesh.ravel(), y_mesh.ravel()))
z1_mesh = X_mesh @ W1 + b1
a1_mesh = sigmoid(z1_mesh)
z2_mesh = a1_mesh @ W2 + b2
z_mesh = z2_mesh.reshape(x_mesh.shape)

fig = go.Figure(data=[
    go.Surface(x=x_mesh, y=y_mesh, z=z_mesh),
    go.Scatter3d(x=X[:, 0], y=X[:, 1], z=y.ravel(), 
                 mode='markers', marker=dict(size=5))
])

fig.update_layout(
    scene = dict(
        xaxis_title='limit',
        yaxis_title='income',
        zaxis_title='balance'
    ),
    width=800,
    height=800
)

fig.show()


## Two features, two neurons

Now let's look at the most complex neural network we will fit today.

<img src="two_features_two_neurons.png" width="1000">


This is the first network where `W1` is a full matrix in both dimensions — previously it had one of its sides equal to 1 (a row vector with one feature and multiple neurons, or a column vector with multiple features and one neuron). Now `W1` is shape `(2, 2)`: rows index the input features, columns index the hidden neurons, and entry `W1[i, j]` is the weight from feature `i` into neuron `j`. 


### Task 9

Train the neural network shown above.


What does this network look like?

In [29]:
x_range = np.linspace(X[:, 0].min(), X[:, 0].max(), 100)
y_range = np.linspace(X[:, 1].min(), X[:, 1].max(), 100)
x_mesh, y_mesh = np.meshgrid(x_range, y_range)

X_mesh = np.column_stack((x_mesh.ravel(), y_mesh.ravel()))
z1_mesh = X_mesh @ W1 + b1
a1_mesh = sigmoid(z1_mesh)
z2_mesh = a1_mesh @ W2 + b2
z_mesh = z2_mesh.reshape(x_mesh.shape)

fig = go.Figure(data=[
    go.Surface(x=x_mesh, y=y_mesh, z=z_mesh),
    go.Scatter3d(x=X[:, 0], y=X[:, 1], z=y.ravel(), 
                 mode='markers', marker=dict(size=5))
])

fig.update_layout(
    scene = dict(
        xaxis_title='limit',
        yaxis_title='income',
        zaxis_title='balance'
    ),
    width=800,
    height=800
)

fig.show()


### Task 10

So far we worked with neural networks that are suited for regression tasks. Write a forward pass of a neural network that is suited for classification.

Also consider and answer the following - do we need to change the procedure for gradient descent? If so, in what ways?

## Recap

You just implemented and trained a bunch of neural networks!

Here are the key components and concepts to remember:
- A feedforward neural network is essentially a series of matrix transformations and nonlinear transformations stacked together.
- Even though we only looked at networks with one hidden layer, we can have as many hidden layers as we like (or as many as your machine allows). Same goes for neurons within each layer.
- Nonlinear activation functions are what allow the neural network to fit complex functions to the data. Sigmoid is one such function, but there are many more.
- The trickiest part (at least conceptually) about deep neural networks is training them, since we have to figure out partial derivatives of the loss function with respect to every parameter, which requires the application of the chain rule.
- We can use these derivatives to calculate the gradient of the loss function with respect to each weight, via the process called backpropagation.
- The process of adjusting weights according to these gradients is called gradient descent - we move weights in the direction that reduces the loss the fastest.
- Learning rate controls the speed of gradient descent.

You do not need to remember the formulas of the derivatives or how to derive them.

